In [2]:
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)
table_panel_name = "working_yearly_with_tfp_wave"

# Export to parquet
df_panel = con.table(table_panel_name).execute()
df_panel.to_parquet(dirs.tmp_dir / f"{table_panel_name}.parquet", index=False)
df_panel.to_csv(dirs.tmp_dir / f"{table_panel_name}.csv", index=False)

In [ ]:
from ast import Tuple
import re
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="model")

input_filepath = dirs.output_dir / "results_3_ivdynamic.txt"
output_filepath = dirs.output_dir / "results_3_ivdyn.tex"

# 1. Parse the text file
with open(input_filepath, 'r') as f:
    lines = f.readlines()

models = {}
current_model = None
model_count = 0
unique_params = set()

for line in lines:
    line = line.strip()
    
    # Detect the start of a new model
    if 'Dynamic panel-data estimation' in line:
        model_count += 1
        current_model = f"dyn{model_count}"
        models[current_model] = {
            'Y': '',
            'params': {}, 
            'obs': '', 
            'groups': '',
            'hansen_stat': '',
            'hansen_p': '',
            'ar1_stat': '',
            'ar1_p': '',
            'ar2_stat': '',
            'ar2_p': ''
        }
        continue
        
    if not current_model:
        continue
        
    # Extract Summary Statistics
    obs_match = re.search(r'Number of obs\s*=\s*(\d+)', line)
    if obs_match:
        models[current_model]['obs'] = f"{int(obs_match.group(1)):,}"
        
    groups_match = re.search(r'Number of groups\s*=\s*(\d+)', line)
    if groups_match:
        models[current_model]['groups'] = f"{int(groups_match.group(1)):,}"

    # Extract Test Results
    hansen_match = re.search(r'Hansen test of overid\. restrictions:.*Prob > Chi2\s*=\s*([\d\.]+)', line)
    if hansen_match:
        p_val = float(hansen_match.group(1))
        models[current_model]['hansen_p'] = f"{p_val:.3f}"
        models[current_model]['hansen_reject'] = p_val < 0.05

    ar1_match = re.search(r'Arellano-Bond test for AR\(1\) in first differences:.*Pr > z\s*=\s*([\d\.]+)', line)
    if ar1_match:
        p_val = float(ar1_match.group(1))
        models[current_model]['ar1_p'] = f"{p_val:.3f}"
        models[current_model]['ar1_reject'] = p_val < 0.05

    ar2_match = re.search(r'Arellano-Bond test for AR\(2\) in first differences:.*Pr > z\s*=\s*([\d\.]+)', line)
    if ar2_match:
        p_val = float(ar2_match.group(1))
        models[current_model]['ar2_p'] = f"{p_val:.3f}"
        models[current_model]['ar2_reject'] = p_val < 0.05

    # Parse the coefficient table
    if line.startswith('|'):
        if 'coef.' in line:
            models[current_model]['Y'] = line.split('|')[1].strip()
            continue
        elif 'coef.' not in line:
            parts = [p.strip() for p in line.split('|')]
            if len(parts) >= 7:
                var_name = parts[1]
                
                # Skip empty lines, dividers, or fixed time effects
                if not var_name or var_name.startswith('year_'):
                    continue
                    
                # Map the constant term
                if var_name == '_con':
                    var_name = 'const'
                    
                coef = parts[2]
                se = parts[3]
                stars = parts[6]
                
                try:
                    # Format numbers to 3 decimal places for a cleaner table
                    coef_fmt = f"{float(coef):.3f}"
                    se_fmt = f"{float(se):.3f}"
                except ValueError:
                    coef_fmt = coef
                    se_fmt = se
                
                models[current_model]['params'][var_name] = {
                    'coef': coef_fmt, 
                    'se': se_fmt, 
                    'stars': stars
                }
                unique_params.add(var_name)

def format_param(param: str) -> str:
    if param == 'const':
        return 'constant'

    d: dict[str, str] = {
        'core': param
    }
    
    if 'ln_' in param:
        d['ln_start'] = 'ln(\\itx{'
        d['ln_end'] = '})'
        d['core'] = d['core'].replace('ln_', '')

    lag_match = re.match(r'L(\d+)\.(.+)', d['core'])
    if lag_match:
        lag_num = lag_match.group(1)
        d['lag'] = f"t-{lag_num}"
        d['core'] = lag_match.group(2)

    # If the core ends with a number
    num_match = re.match(r'(.+?)(\d+)$', d['core'])
    if num_match:
        d['core'] = num_match.group(1)
        d['num'] = num_match.group(2)

    # If the core ends with _i, _j, or _k
    i_match = re.match(r'(.+?)_([ijk])$', d['core'])
    if i_match:
        d['core'] = i_match.group(1)
        d['index'] = i_match.group(2)
    
    insub_props = ['num']
    insub_v = [d.get(prop) for prop in insub_props if d.get(prop) is not None]
    has_insubscript = len(insub_v) > 0
    insubscript_str = f"\\textsubscript{{${','.join(insub_v)}$}}" if has_insubscript else ''

    outsub_props = ['index', 'lag']
    outsub_v = [d.get(prop) for prop in outsub_props if d.get(prop) is not None]
    has_subscript = len(outsub_v) > 0
    outsubscript_str = f"\\textsubscript{{${','.join(outsub_v)}$}}" if has_subscript else ''
    
    final_str = "".join([
        d.get('ln_start', ''),
        f"\\itx{{{d.get('core', '')}}}",
        insubscript_str,
        d.get('ln_end', ''),
        outsubscript_str
    ])
    return final_str.replace('_', '\\_')

def form_block(args: tuple[str | None, str | None] | None = None, type: str = 'l', size: tuple[str | None, str | None] = (None, None)) -> str:
    param_str, value_str = args if args else (None, None)
    if not param_str:
        param_str = '~'
    if not value_str:
        value_str = '~'
    if size[0]:
        param_str = f"\\{size[0]}{{{param_str}}}"
    if size[1]:
        value_str = f"\\{size[1]}{{{value_str}}}"
    return (
        f"\t\t\\{type}block{{\n"
        f"\t\t\t{param_str}\n"
        f"\t\t\t\\\\\n"
        f"\t\t\t{value_str}\n"
        f"\t\t}}"
    )

# 2. Sort Parameters (Force 'const' to the top)
sorted_params = sorted(list(unique_params), key=lambda x: (x != 'const', x))

# 3. Generate the LaTeX Table
model_names = list(models.keys())

latex_lines = []
latex_lines.append(f"\t\\begin{{tabular}}{{l{'c' * len(model_names)}}}")

# Header row
cols = []
lblock = form_block()
cols.append(lblock)
for i, mod in enumerate(model_names):
    y_str = format_param(models[mod]['Y'])
    cblock = form_block((f"({i + 1})", y_str), type='c', size=('footnotesize', 'small'))
    cols.append(cblock)
header = " & ".join(cols) + " \\\\[0.8em]"
latex_lines.append("\t\t\\toprule\\toprule")
latex_lines.append(header)
latex_lines.append("\t\t\\toprule")

# Parameter rows
for idx, param in enumerate(sorted_params):
    row_lines = []
    lblock = form_block((format_param(param), '~'))
    row_lines.append(lblock)
    
    # Build the \cblock for each model's estimates
    for mod in model_names:
        mod_data = models[mod]['params'].get(param)
        if mod_data:
            star_str = mod_data['stars']
            cblock = form_block(
                (f"${mod_data['coef']}$", f"({mod_data['se']}){star_str}"),
                type='c',
                size=(None, 'footnotesize')
            )
            row_lines.append(cblock)
        else:
            row_lines.append(" ") # Blank cell if parameter is not in this model
            
    # Join the row blocks
    latex_lines.append(" & ".join(row_lines))
    latex_lines.append("\t\t\\\\ [0.9em]")

# Test Results Section
latex_lines.append("\t\t\\hline")

def build_test_cblock(reject: bool | None, p_val: str) -> str:
    if reject is None or not p_val:

        #
        return " "
    symbol = "\\checkmark" if reject else "\\text{\\sffamily X}"
    # Generate stars based on p_val
    stars = ""
    p_thresholds = [0.01, 0.05, 0.1]
    p_float = float(p_val)
    for threshold in p_thresholds:
        if p_float < threshold:
            stars += "*"
    star_str = f"{stars}" if stars else ""
    return form_block(
        (symbol, f"({p_val}){star_str}"),
        type='c',
        size=(None, 'footnotesize')
    )

# Hansen Overidentification Test
hansen_cells = [
    build_test_cblock(models[m].get('hansen_reject'), models[m]['hansen_p'])
    for m in model_names
]
hansen_row = f"\t\t{form_block(('Hansen Test', None))} & " + " & ".join(hansen_cells) + "\n\t\t\\\\ [0.9em]"
latex_lines.append(hansen_row)

# Arellano-Bond AR(1) Test
ar1_cells = [
    build_test_cblock(models[m].get('ar1_reject'), models[m]['ar1_p'])
    for m in model_names
]
ar1_row = f"\t\t{form_block(('AR(1) Test', None))} & " + " & ".join(ar1_cells) + "\n\t\t\\\\ [0.9em]"
latex_lines.append(ar1_row)

# Arellano-Bond AR(2) Test
ar2_cells = [
    build_test_cblock(models[m].get('ar2_reject'), models[m]['ar2_p'])
    for m in model_names
]
ar2_row = f"\t\t{form_block(('AR(2) Test', None))} & " + " & ".join(ar2_cells) + "\n\t\t\\\\ [0.9em]"
latex_lines.append(ar2_row)

# Summary Statistics rows
latex_lines.append("\t\t\\hline")

i_row = "\t\ti & " + " & ".join([models[m]['groups'] for m in model_names]) + "\n\t\t\\\\"
latex_lines.append(i_row)

obs_row = "\t\tObservations & " + " & ".join([models[m]['obs'] for m in model_names]) + "\n\t\t\\\\"
latex_lines.append(obs_row)

latex_lines.append("\t\t\\bottomrule")
latex_lines.append("\t\\end{tabular}")

# 4. Write to file
with open(output_filepath, 'w', encoding='utf-8') as f:
    f.write("\n".join(latex_lines))
    
print(f"Successfully generated LaTeX table with {len(model_names)} models at {output_filepath}")

Successfully generated LaTeX table with 3 models at C:\Users\lazyst\Files\ucl\Dissertation\model\output\results_3_ivdyn.tex
